# CVPR 2026 — Video demos (run in Colab)

**Runtime → Change runtime type → T4 GPU** (or L4/A100 with Pro), then run cells top to bottom.

Four demos on video: **depth per frame**, **VLM scene-change Q&A**, **SAM2 object tracking**, **text→video**.

*(Built as a notebook because a browser Colab kernel is far more reliable than the CLI for multi-model pipelines.)*

In [ ]:
# 1) Setup + build a valid two-scene clip (meadow -> underwater), with fallback
!pip -q install transformers imageio imageio-ffmpeg qwen-vl-utils decord diffusers accelerate
import os, subprocess
def run(c): return subprocess.run(c, shell=True, capture_output=True, text=True).returncode
srcs=[('a.mp4','https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/360/Big_Buck_Bunny_360_10s_1MB.mp4'),
      ('b.mp4','https://test-videos.co.uk/vids/jellyfish/mp4/h264/360/Jellyfish_360_10s_1MB.mp4')]
for n,u in srcs:
    if not os.path.exists(n) or os.path.getsize(n)<100000: run(f'wget -q --timeout=40 -O {n} {u}')
def enc(src,out,color):
    if os.path.exists(src) and os.path.getsize(src)>100000:
        run(f'ffmpeg -y -loglevel error -i {src} -t 8 -vf scale=480:270,setsar=1,fps=24 -an -c:v libx264 -pix_fmt yuv420p {out}')
    else:
        run(f'ffmpeg -y -loglevel error -f lavfi -i color=c={color}:s=480x270:r=24:d=8 -pix_fmt yuv420p {out}')
enc('a.mp4','a2.mp4','forestgreen'); enc('b.mp4','b2.mp4','navy')
open('list.txt','w').write("file 'a2.mp4'\nfile 'b2.mp4'\n")
run('ffmpeg -y -loglevel error -f concat -safe 0 -i list.txt -c copy clip.mp4')
import imageio
print('clip.mp4 frames:', len([f for f in imageio.get_reader('clip.mp4')]))

In [ ]:
# 2) Video depth — Depth Anything per frame (original | depth)
import imageio, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline
frames=[f for f in imageio.get_reader('clip.mp4')]
dp=pipeline('depth-estimation',model='depth-anything/Depth-Anything-V2-Small-hf',device=0)
out=[]
for i in range(0,len(frames),2):
    im=Image.fromarray(frames[i]).convert('RGB').resize((320,180))
    d=np.array(dp(im)['depth'],np.float32); dn=(d-d.min())/(d.max()-d.min()+1e-6)
    rgb=(plt.get_cmap('inferno')(dn)[:,:,:3]*255).astype(np.uint8)
    out.append(np.concatenate([np.array(im),rgb],1))
imageio.mimsave('depth_video.mp4',out,fps=12)
from IPython.display import Video; Video('depth_video.mp4',embed=True,width=680)

In [ ]:
# 3) Video understanding — ask about the SIGNIFICANT CHANGE (temporal Q a still model can't answer)
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
m=Qwen2VLForConditionalGeneration.from_pretrained('Qwen/Qwen2-VL-2B-Instruct',torch_dtype=torch.float16,device_map='cuda')
proc=AutoProcessor.from_pretrained('Qwen/Qwen2-VL-2B-Instruct')
for q in ['What is happening in this video? Describe it in 2-3 sentences.',
          'What is the single most significant change or event that happens over the course of the video?']:
    msg=[{'role':'user','content':[{'type':'video','video':'clip.mp4','fps':1.5,'max_pixels':360*640},{'type':'text','text':q}]}]
    txt=proc.apply_chat_template(msg,tokenize=False,add_generation_prompt=True)
    res=process_vision_info(msg)
    inp=proc(text=[txt],images=res[0],videos=res[1],padding=True,return_tensors='pt').to('cuda')
    o=m.generate(**inp,max_new_tokens=160)
    print('Q:',q); print('A:',proc.batch_decode(o[:,inp.input_ids.shape[1]:],skip_special_tokens=True)[0].strip(),'\n')

In [ ]:
# 4) Object tracking with SAM2 — click one point on frame 0, track it through the clip
!pip -q install 'git+https://github.com/facebookresearch/sam2.git'
!rm -rf frames && mkdir -p frames && ffmpeg -y -loglevel error -i clip.mp4 -vf scale=480:270 -q:v 2 -start_number 0 'frames/%05d.jpg'
import torch, numpy as np, imageio, glob
from PIL import Image
from sam2.sam2_video_predictor import SAM2VideoPredictor
predictor=SAM2VideoPredictor.from_pretrained('facebook/sam2.1-hiera-small')
with torch.inference_mode(), torch.autocast('cuda',dtype=torch.bfloat16):
    state=predictor.init_state('frames')
    predictor.add_new_points_or_box(state,frame_idx=0,obj_id=1,points=np.array([[240,150]],np.float32),labels=np.array([1],np.int32))
    seg={}
    for fidx,ids,logits in predictor.propagate_in_video(state):
        seg[fidx]=(logits[0]>0).cpu().numpy()[0]
files=sorted(glob.glob('frames/*.jpg')); ov=[]
for i,f in enumerate(files):
    im=np.array(Image.open(f).convert('RGB')).astype(float)
    if i in seg:
        mm=seg[i]
        if mm.shape!=im.shape[:2]:
            mm=np.array(Image.fromarray(mm).resize((im.shape[1],im.shape[0])))>0
        im[mm]=im[mm]*0.4+np.array([255,64,64])*0.6
    ov.append(im.astype('uint8'))
imageio.mimsave('track.mp4',ov,fps=12)
from IPython.display import Video; Video('track.mp4',embed=True,width=520)

In [ ]:
# 5) Text -> video — generate a clip from a prompt (heaviest; L4/A100 comfortable, T4 works)
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video
pipe=DiffusionPipeline.from_pretrained('damo-vilab/text-to-video-ms-1.7b',torch_dtype=torch.float16,variant='fp16')
pipe.scheduler=DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
frames=pipe('a cinematic drone shot flying over a snowy mountain range at sunrise',num_frames=24,num_inference_steps=25).frames[0]
export_to_video(frames,'gen_video.mp4',fps=8)
from IPython.display import Video; Video('gen_video.mp4',embed=True,width=512)